> **Trust but verify.** Even examples Rick shares and AI-generated suggestions can be wrong or unsafe. Read code, commands and configuration before running them; use disposable copies, keep backups, and verify results. Don't let an example eat your homework.

# From processes to source and open-source projects

Use this notebook after the [instrumented PyTorch example](instrumented-pytorch.ipynb), with the same [environment](setup.md). Identify the running pieces, distinguish saved output from live state, then map the processes and model calls to source and upstream projects. No model download or inference is needed here.

Each notebook can have its own kernel. These observations describe this notebook's kernel, not automatically the one running the PyTorch example. The diagram describes the model demonstration; opening this notebook does not load that model.

You can follow by watching Rick's screen without an installation or account. The core route is the process map, one model-call source link and the upstream project table. The detailed source table is optional reference within the first-half walkthrough.


## 0. Find the pieces doing the work

In this native laptop example, [JupyterLab](https://jupyterlab.readthedocs.io/en/4.4.x/user/documents_kernels.html) renders the notebook interface in the browser. [Jupyter Server](https://jupyter-server.readthedocs.io/en/stable/developers/architecture.html) serves files, manages kernels and relays messages. The Python kernel runs our cells in a separate process. PyTorch is a library used by that process; it is not a Jupyter component.

| Piece | Role here | Evidence to inspect |
| --- | --- | --- |
| Notebook document | Saved cells, metadata and outputs | A disposable `.ipynb` copy opened as text |
| JupyterLab | Interface in your browser | Notebook tab, kernel indicator and About dialog |
| Jupyter Server | File service, kernel lifecycle and message relay | The terminal that launched JupyterLab |
| ipykernel / IPython | Executes Python cells and keeps live variables | Shell/kernel class and process ID below |
| Python environment | Supplies the interpreter and installed packages | Executable and environment prefix below |
| PyTorch | Performs tensor operations and model execution | Imported module/version below; device tables in the PyTorch notebook |
| Model and weights | Tokenizer, BERT configuration and trained parameters | Pinned source/revision and model identity in the PyTorch notebook |

The diagnostics read a small set of local values. They do not inspect connection files, server credentials or environment-variable contents.


In [ ]:
from html import escape
from IPython.display import HTML, display


def show_table(headers, rows):
    """Render small observations without adding a table package."""
    def row(values, tag):
        return "<tr>" + "".join(
            f"<{tag}>{escape(str(value))}</{tag}>" for value in values
        ) + "</tr>"
    display(HTML("<table>" + row(headers, "th")
                 + "".join(row(values, "td") for values in rows) + "</table>"))


In [ ]:
import os
import sys
from importlib.metadata import PackageNotFoundError, version
from IPython import get_ipython
import torch

shell = get_ipython()
kernel = getattr(shell, "kernel", None)

def class_name(value):
    return (f"{type(value).__module__}.{type(value).__name__}"
            if value is not None else "Not present in this execution context")

show_table(["Live observation", "Value"], [
    ("Python executable", sys.executable),
    ("Environment prefix", sys.prefix),
    ("Python version", sys.version.split()[0]),
    ("Process PID", os.getpid()),
    ("Parent PID (may be server or launcher)", os.getppid()),
    ("IPython shell class", class_name(shell)),
    ("Kernel class", class_name(kernel)),
    ("Imported PyTorch version", torch.__version__),
    ("Imported PyTorch module", torch.__file__),
])

installed_packages = []
for package in ("jupyterlab", "jupyter-server", "ipykernel", "ipython", "torch", "transformers"):
    try:
        installed_packages.append((package, version(package)))
    except PackageNotFoundError:
        installed_packages.append((package, "Not installed in this Python environment"))
show_table(["Package in the kernel environment", "Installed version"], installed_packages)
print(f"Optional macOS/Linux terminal check: ps -p {os.getpid()},{os.getppid()} -o pid,ppid,comm")


### Compare the notebook observations with the running interface

Look at the browser address bar and JupyterLab's About dialog. In the terminal where you launched JupyterLab, find the Jupyter Server version and host/port from its startup messages. If you share these observations, omit any URL token. The optional `ps` command above shows process names, not argument lists; a parent process may be a launcher rather than the server itself.

The package table describes packages installed in **this kernel's Python environment**. It does not discover the active server or prove which interface the browser is using. A server and a kernel can use different environments. Here, compare what you actually launched with the executable/prefix and UI evidence before saying they match.

Open a disposable notebook copy as text and find `cells`, `source`, `outputs` and `metadata`. Those are fields in the [notebook document](https://nbformat.readthedocs.io/en/latest/format_description.html), not the Python variables holding our model.

When you press Run, the interface sends an execution request through the server to the kernel. The kernel calls Python/PyTorch, then sends results back through the server for display. Saving records notebook content and selected outputs; it does not preserve the loaded Python model object.



## Separate the notebook document from live kernel state

Run the next cell twice: its counter increases while the same kernel is alive. Save your disposable notebook, then choose **Kernel → Restart Kernel** without running all cells. Previously rendered or saved output can remain visible even though the old Python variables are gone.

Run only the next cell after the restart: the counter starts at 1. Compare the PID before and after the restart; it normally changes, although operating systems can reuse PIDs. This mapping notebook does not load `model`, so its model-presence line normally reports `False`, even before a restart. If both notebooks deliberately share a kernel, it instead reflects that shared namespace. Finally restart and run all cells to restore this notebook's observations. Closing a notebook tab alone does not necessarily stop its kernel; inspect JupyterLab's [Running panel](https://jupyterlab.readthedocs.io/en/4.4.x/user/running.html).


In [ ]:
import os

state_probe = globals().get("state_probe", 0) + 1
print("Runs of this probe in the current kernel:", state_probe)
print("Python process PID:", os.getpid())
print("Model present in live variables:", "model" in globals())


## Map the processes to code and projects

The browser runs JupyterLab's JavaScript interface outside the conda environment. The environment supplies installed packages and an interpreter used to launch two distinct processes: server and kernel. It is not a process container or a security boundary. This is the demonstrated native setup, not a claim that every Jupyter deployment uses one environment. Kernel package metadata alone cannot prove which environment serves the browser.

Run sends a request through the server to the kernel. IPython executes the cell; our Transformers model uses PyTorch tensor operations on CPU. Kernel replies and output messages return through the server to the interface. The arrows summarize this path, not a complete network-protocol diagram or Python stack trace. The server also saves notebook documents, which is separate from model execution. Downloaded weights live in a cache outside the scope of the environment box and are loaded into kernel memory; they are not bundled in our environment definition.

```mermaid
flowchart TD
    subgraph host["Laptop: native demonstration"]
        subgraph browser["Browser process"]
            lab["JupyterLab / notebook UI"]
        end
        subgraph env["conda environment: jupyter-pytorch-week2"]
            subgraph server["Server process"]
                js["Jupyter Server"]
            end
            subgraph kernel["Separate kernel / Python process"]
                ipk["ipykernel: execution requests"]
                ipy["IPython: Python cells and live state"]
                bert["Transformers: BERT model / loaded weights"]
                torch["PyTorch: tensor operations on CPU"]
                ipk --> ipy
                ipy --> bert
                bert <--> torch
            end
        end
        lab -->|"Run: code request"| js
        js -->|"Execute request"| ipk
        ipk -->|"Replies / output messages"| js
        js -->|"Results for display"| lab
    end
```

| Piece we observed | Upstream project | What to find there |
| --- | --- | --- |
| JupyterLab interface | [jupyterlab/jupyterlab](https://github.com/jupyterlab/jupyterlab) | Notebook UI and browser-side kernel requests |
| Server process | [jupyter-server/jupyter_server](https://github.com/jupyter-server/jupyter_server) | File service, kernel management and message relay |
| Kernel process | [ipython/ipykernel](https://github.com/ipython/ipykernel) | Python kernel's execution handler |
| IPython shell in the kernel | [ipython/ipython](https://github.com/ipython/ipython) | Cell execution and live user namespace |
| Python interpreter | [python/cpython](https://github.com/python/cpython) | Language runtime used by these Python processes |
| Environment tooling | [conda/conda](https://github.com/conda/conda) | Environment/package management; conda is not the kernel |
| PyTorch module and CPU tensors | [pytorch/pytorch](https://github.com/pytorch/pytorch) | Tensor operations and module dispatch; this is not a Jupyter component |
| BERT model class and tokenizer | [huggingface/transformers](https://github.com/huggingface/transformers) | Architecture/tokenization code built on PyTorch |
| Downloaded checkpoint | [Pinned model files and card](https://huggingface.co/M-FAC/bert-tiny-finetuned-sst2/tree/41ad6709ec46b414749b37daf49cf5ca1c7dba7c) | Trained weights/configuration, separate from library code; no license inference from the library |
| Saved notebook format | [jupyter/nbformat](https://github.com/jupyter/nbformat) | Schema for cells, metadata and saved outputs |

The [Mermaid source](images/component-map.mmd) is retained for editing.


## Selective source path for Run and prediction

These source files were read at the release tags below and checked against the workspace versions. JupyterLab 4.4.9 and Server 2.21.1 are the earlier clean conda validation versions; the current scratch kernel has ipykernel 6.30.1 and IPython 9.17.1. IPython is a transitive dependency, not a new pin. Exact Mac patch versions were not collected. Compare the notebook observations and server terminal before assuming your installation matches these source snapshots.

| Visible action / boundary | Selected implementation | What the source establishes |
| --- | --- | --- |
| Execute a code cell in the interface | [`CodeCell.execute`, JupyterLab 4.4.9](https://github.com/jupyterlab/jupyterlab/blob/v4.4.9/packages/cells/src/widget.ts#L1734-L1785), then [`OutputArea.execute`](https://github.com/jupyterlab/jupyterlab/blob/v4.4.9/packages/outputarea/src/widget.ts#L934-L970) | Creates the execution/output handling for the cell and calls the kernel client |
| Browser-side kernel client | [`KernelConnection.requestExecute`](https://github.com/jupyterlab/jupyterlab/blob/v4.4.9/packages/services/src/kernel/default.ts#L786-L813) | Constructs an `execute_request` message, not local Python execution |
| Server relay in both directions | [`ZMQChannelsWebsocketConnection.handle_incoming_message` / `handle_outgoing_message`, Server 2.21.1](https://github.com/jupyter-server/jupyter_server/blob/v2.21.1/jupyter_server/services/kernels/connection/channels.py#L516-L589) | Relays between browser WebSocket messages and kernel ZeroMQ channels |
| Kernel execution request | [`Kernel.execute_request`, ipykernel 6.30.1](https://github.com/ipython/ipykernel/blob/v6.30.1/ipykernel/kernelbase.py#L721-L796) | Invokes the Python kernel's `do_execute` and prepares the reply |
| Execute Python cell code | [`IPythonKernel.do_execute`](https://github.com/ipython/ipykernel/blob/v6.30.1/ipykernel/ipkernel.py#L370-L478); [`InteractiveShell.run_cell` / `run_cell_async`, IPython 9.17.1](https://github.com/ipython/ipython/blob/9.17.1/IPython/core/interactiveshell.py#L3202-L3390) | Selects synchronous/asynchronous execution depending on context; these are branches, not an unconditional stack |
| `with torch.inference_mode()` | [`inference_mode`, PyTorch 2.8.0](https://github.com/pytorch/pytorch/blob/v2.8.0/torch/autograd/grad_mode.py#L212-L280) | Disables autograd-related bookkeeping for inference; `model.eval()` separately sets evaluation behavior |
| `model(**inputs)` | [`Module._wrapped_call_impl` / `_call_impl`](https://github.com/pytorch/pytorch/blob/v2.8.0/torch/nn/modules/module.py#L1769-L1790), then [`BertForSequenceClassification.forward`, Transformers 4.56.2](https://github.com/huggingface/transformers/blob/v4.56.2/src/transformers/models/bert/modeling_bert.py#L1460-L1507) | PyTorch dispatches to the model's forward method; BERT produces pooled output and the classifier produces logits |
| `torch.softmax(logits, dim=-1)` | [Native `softmax`, PyTorch 2.8.0](https://github.com/pytorch/pytorch/blob/v2.8.0/aten/src/ATen/native/SoftMax.cpp#L411-L423) | Normalizes along the selected dimension and delegates to the device implementation; scores are not calibrated confidence |
| Instrumented stage callbacks | [`Module.register_forward_hook`](https://github.com/pytorch/pytorch/blob/v2.8.0/torch/nn/modules/module.py#L1676-L1759) and [`_call_impl` hook handling](https://github.com/pytorch/pytorch/blob/v2.8.0/torch/nn/modules/module.py#L1777-L1860) | Hooks observe module results; returning `None` leaves them unchanged |

These are selected implementation entry points, not every internal call frame. The model-library architecture and downloaded trained weights have different sources. The instrumented notebook's callbacks observe outputs and return `None`; its `finally` block removes their handles.


## Finish the mapping

Use the process observations, source links and repository table to locate where each piece is developed. Stop here for review.
